In [1]:
# Block 1 (Updated): Free / API-Key-Free Model Setup (Using HuggingFace / Transformers)
!pip install -qU langgraph langchain-core langchain-community transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 250.2/250.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 572.1/572.1 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 31.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
# Block 1: Imports and LLM Pipeline Setup
import operator
from typing import TypedDict, Annotated
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
from langgraph.graph import StateGraph, START, END

# Free Light-weight Model
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")

pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=pipe)

/tmp/ipykernel_1821/634477792.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.llms import HuggingFacePipeline


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
/tmp/ipykernel_1821/634477792.py:21: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFacePipeline``.
  llm = HuggingFacePipeline(pipeline=pipe)


In [3]:
# Block 2: State Definition
# Is State object mein har parallel node apna output specific key par ya combined array mein push kar sakta hai.

class ReviewState(TypedDict):
    review: str
    sentiment: str
    issues: str
    intent: str
    # Aggregate list to hold combined summary
    summary_reports: Annotated[list, operator.add]

In [4]:
# Block 3: Define 3 Parallel Worker Nodes & 1 Aggregator Node

def node_sentiment(state: ReviewState):
    """Parallel Node 1: Review ka sentiment determine karega"""
    prompt = f"<|user|>\nAnalyze the sentiment (Positive, Negative, or Neutral) of this product review: '{state['review']}'. Return only the sentiment.\n<|assistant|>\n"
    response = llm.invoke(prompt).replace(prompt, "").strip()
    return {
        "sentiment": response,
        "summary_reports": [f"[SENTIMENT ANALYSIS]\n{response}"]
    }

def node_issues(state: ReviewState):
    """Parallel Node 2: Key product issues/flaws identify karega"""
    prompt = f"<|user|>\nExtract main product issues or complaints from this review: '{state['review']}'. Keep it bulleted.\n<|assistant|>\n"
    response = llm.invoke(prompt).replace(prompt, "").strip()
    return {
        "issues": response,
        "summary_reports": [f"[KEY ISSUES]\n{response}"]
    }

def node_intent(state: ReviewState):
    """Parallel Node 3: Customer intent (Refund/Exchange/Feedback) classify karega"""
    prompt = f"<|user|>\nIdentify the customer's intent (e.g., Refund, Replacement, General Feedback) from this review: '{state['review']}'.\n<|assistant|>\n"
    response = llm.invoke(prompt).replace(prompt, "").strip()
    return {
        "intent": response,
        "summary_reports": [f"[CUSTOMER INTENT]\n{response}"]
    }

def aggregator_node(state: ReviewState):
    """Fan-in Node: Teeno parallel nodes ki report compile karega"""
    return state

In [5]:
# Block 4: Connect Parallel Edges (Fan-Out) and Aggregator Edge (Fan-In)
builder = StateGraph(ReviewState)

# 1. Add Nodes
builder.add_node("sentiment_analyzer", node_sentiment)
builder.add_node("issues_extractor", node_issues)
builder.add_node("intent_classifier", node_intent)
builder.add_node("aggregator", aggregator_node)

# 2. Parallel Branching (Fan-Out):
# START node se teenon worker nodes par parallel execution start hoga
builder.add_edge(START, "sentiment_analyzer")
builder.add_edge(START, "issues_extractor")
builder.add_edge(START, "intent_classifier")

# 3. Parallel Merging (Fan-In):
# Teeno nodes apne execution end par aggregator node me merge honge
builder.add_edge("sentiment_analyzer", "aggregator")
builder.add_edge("issues_extractor", "aggregator")
builder.add_edge("intent_classifier", "aggregator")

# 4. Final Transition
builder.add_edge("aggregator", END)

# Graph Compile karein
review_graph = builder.compile()

In [6]:
# Block 5: Test the Workflow with a Sample Customer Review
sample_review = (
    "The battery life on this laptop is terrible, it dies in 2 hours! "
    "Also, the charger stopped working on the second day. "
    "I am very unhappy and want a full refund immediately."
)

# Run the graph with a fresh initial state
input_state = {
    "review": sample_review,
    "sentiment": "",
    "issues": "",
    "intent": "",
    "summary_reports": []
}

output = review_graph.invoke(input_state)

# Display the aggregated parallel outputs
print("=" * 50)
print("PRODUCT REVIEW ANALYZER - SUMMARY REPORT")
print("=" * 50)
print(f"Customer Review: \"{output['review']}\"\n")

for report in output["summary_reports"]:
    print(report)
    print("-" * 50)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE

PRODUCT REVIEW ANALYZER - SUMMARY REPORT
Customer Review: "The battery life on this laptop is terrible, it dies in 2 hours! Also, the charger stopped working on the second day. I am very unhappy and want a full refund immediately."

[CUSTOMER INTENT]
The customer's intent from this review is "Refund" based on the specific phrases "The battery life on this laptop is terrible" and "The charger stopped working on the second day."
--------------------------------------------------
[KEY ISSUES]
- The battery life on this laptop is terrible, it dies in 2 hours!
- The charger stopped working on the second day.
- I am very unhappy and want a full refund immediately.

Remember to use bullet points to break up the text and make it easier to read and understand.
--------------------------------------------------
[SENTIMENT ANALYSIS]
The sentiment of this product review is negative. The reviewer has expressed their disappointment with the battery life and the lack of a reliable charger. They are u

In [7]:
# Block 5 (Positive Review Test): Run Workflow with a Positive Customer Review
sample_positive_review = (
    "Absolutely love this laptop! The screen quality is fantastic, "
    "and the battery lasts all day. Highly recommended for students!"
)

# Run graph with fresh initial state
input_state = {
    "review": sample_positive_review,
    "sentiment": "",
    "issues": "",
    "intent": "",
    "summary_reports": []
}

output = review_graph.invoke(input_state)

# Display the aggregated parallel outputs
print("=" * 50)
print("PRODUCT REVIEW ANALYZER - POSITIVE REVIEW REPORT")
print("=" * 50)
print(f"Customer Review: \"{output['review']}\"\n")

for report in output["summary_reports"]:
    print(report)
    print("-" * 50)

[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


PRODUCT REVIEW ANALYZER - POSITIVE REVIEW REPORT
Customer Review: "Absolutely love this laptop! The screen quality is fantastic, and the battery lasts all day. Highly recommended for students!"

[CUSTOMER INTENT]
The customer's intent from this review is "Highly recommended for students!" indicating that they are satisfied with the laptop's screen quality and battery life. This is consistent with the intent expressed in the review, which is to recommend the laptop to others who are looking for a high-quality laptop for students.
--------------------------------------------------
[KEY ISSUES]
- Absolutely love this laptop! The screen quality is fantastic, and the battery lasts all day. Highly recommended for students!
- The laptop's screen quality is excellent, and the battery life is impressive. It's highly recommended for students.
--------------------------------------------------
[SENTIMENT ANALYSIS]
The sentiment of this product review is positive. The reviewer has expressed their 

In [8]:
import IPython.display as display

# HTML/JS UI Code embedded directly in Colab
html_code = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <script src="https://cdn.tailwindcss.com"></script>
</head>
<body class="bg-slate-900 text-slate-100 p-6 min-h-screen">
  <div class="max-w-xl mx-auto bg-slate-800 border border-slate-700 rounded-xl p-6 shadow-xl">
    <h2 class="text-xl font-bold text-indigo-400 mb-2">🛍️ Review Sentiment & Analyzer</h2>
    <textarea id="reviewInput" class="w-full bg-slate-900 border border-slate-700 rounded-lg p-3 text-sm focus:outline-none focus:border-indigo-500 text-white" rows="3" placeholder="Type your review here..."></textarea>

    <div class="flex gap-2 mt-3">
      <button onclick="analyze()" class="bg-indigo-600 hover:bg-indigo-500 text-white px-4 py-2 rounded-lg text-sm font-semibold w-full">Analyze Review</button>
      <button onclick="setPos()" class="bg-emerald-800 text-emerald-200 px-3 py-2 rounded-lg text-xs">Positive</button>
      <button onclick="setNeg()" class="bg-rose-800 text-rose-200 px-3 py-2 rounded-lg text-xs">Negative</button>
    </div>

    <div id="result" class="mt-5 hidden border-t border-slate-700 pt-4">
      <div class="flex justify-between items-center mb-3">
        <span class="text-xs text-slate-400 font-semibold">SENTIMENT DETECTED:</span>
        <span id="sentimentBadge" class="px-3 py-1 text-xs font-bold rounded-full"></span>
      </div>
      <div class="bg-slate-900 p-3 rounded-lg border border-slate-700/50">
        <div class="text-xs text-indigo-400 font-semibold mb-1">Key Summary</div>
        <p id="summaryText" class="text-xs text-slate-300"></p>
      </div>
    </div>
  </div>

  <script>
    function setPos() {
      document.getElementById('reviewInput').value = "This laptop is amazing! Battery life is awesome and display is crystal clear.";
    }
    function setNeg() {
      document.getElementById('reviewInput').value = "The charger broke on second day and battery dies in 1 hour. Horrible experience!";
    }
    function analyze() {
      const text = document.getElementById('reviewInput').value.toLowerCase();
      if(!text) return;

      const res = document.getElementById('result');
      const badge = document.getElementById('sentimentBadge');
      const summary = document.getElementById('summaryText');
      res.classList.remove('hidden');

      if (text.includes("amazing") || text.includes("awesome") || text.includes("great") || text.includes("love") || text.includes("good")) {
        badge.className = "px-3 py-1 text-xs font-bold rounded-full bg-emerald-500/20 text-emerald-400 border border-emerald-500/30";
        badge.innerText = "🟢 POSITIVE";
        summary.innerText = "Customer loved the overall experience and performance of the product.";
      } else if (text.includes("horrible") || text.includes("broke") || text.includes("terrible") || text.includes("bad") || text.includes("dies")) {
        badge.className = "px-3 py-1 text-xs font-bold rounded-full bg-rose-500/20 text-rose-400 border border-rose-500/30";
        badge.innerText = "🔴 NEGATIVE";
        summary.innerText = "Hardware defects and short battery performance reported by customer.";
      } else {
        badge.className = "px-3 py-1 text-xs font-bold rounded-full bg-slate-500/20 text-slate-300 border border-slate-500/30";
        badge.innerText = "⚪ NEUTRAL";
        summary.innerText = "General feedback without strong positive or negative statements.";
      }
    }
  </script>
</body>
</html>
"""

display.display(display.HTML(html_code))